#Silver Stage
**Salesperson:**
Creating a new data frame, to cleanse the raw data, and make the necessary transformations.

In [0]:
df_salesperson_clean = spark.read.csv("/Volumes/final_project/bronze/data/Salesperson.csv", inferSchema=True, header=True)
display(df_salesperson_clean)

Fixing the delimiter issue (entries separated by tab) in order to create the appropriate columns.

In [0]:
df_salesperson_clean = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", "\t") \
    .load("/Volumes/final_project/bronze/data/Salesperson.csv")

In [0]:
df_salesperson_clean.display()

In [0]:
df_salesperson_clean.printSchema()

Changing the data types for Employee Key and Employee ID because they should be numerical.

In [0]:
from pyspark.sql.functions import col

df_salesperson_clean = df_salesperson_clean.withColumn("EmployeeKey", col("EmployeeKey").cast("int"))
df_salesperson_clean = df_salesperson_clean.withColumn("EmployeeID", col("EmployeeID").cast("long"))

In [0]:
df_salesperson_clean.printSchema()

Changing the column names.

In [0]:
df_salesperson_clean = df_salesperson_clean.toDF(
    "employee_key",
    "employee_id",
    "salesperson",
    "title",
    "upn"
)
df_salesperson_clean.display()

Optional: creating a temp view to test functionality.

In [0]:
df_salesperson_clean.createOrReplaceTempView("salesperson_view")

In [0]:
spark.sql("SELECT * FROM salesperson_view WHERE employee_key > 280").display();

Filtering the emails to make sure they follow the suggested format.

In [0]:
from pyspark.sql.functions import col

invalid_emails = df_salesperson_clean.filter(
    ~col("upn").rlike("^[A-Za-z0-9._%+-]+@adventureworks\\.com$")
)

invalid_emails.show()

In [0]:
from pyspark.sql.functions import trim, col

df_salesperson_clean = df_salesperson_clean.withColumn(
    "salesperson",
    trim(col("salesperson"))
)
df_salesperson_clean = df_salesperson_clean.withColumn(
    "title",
    trim(col("title"))
)
df_salesperson_clean.display()


Checking to see if there are null values for employee key & ID.

In [0]:
df_salesperson_clean.filter(col("employee_key").isNull()).display()

In [0]:
df_salesperson_clean.filter(col("employee_id").isNull()).display()

Checking to see if there are duplicates for employee key and employee ID.

In [0]:
df_salesperson_clean.groupBy("employee_id").count().filter("count > 1").display()

In [0]:
df_salesperson_clean.groupBy("employee_key").count().filter("count > 1").display()

Creating the delta table Salesperson.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS final_project.silver")

df_salesperson_clean.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("final_project.silver.salesperson")

display(spark.table("final_project.silver.salesperson"))

